# 🌿 DigitalWell AI — Model Development

This notebook contains the core machine-learning workflow for the DigitalWell AI datathon prototype, explored on IBM Z / LinuxONE (s390x).

## Workflow
1. Load and inspect the smartphone-usage dataset.
2. Prepare features and train a balanced Logistic Regression pipeline.
3. Evaluate on a stratified 20% holdout set.
4. Inspect permutation feature importance.
5. Compare hypothetical screen-time, sleep, and social-media scenarios.
6. Export the fitted pipeline and feature metadata for the Streamlit dashboard.

## Dataset requirement
Place Smartphone_Usage_And_Addiction_Analysis_7500_Rows.csv beside this notebook before running it. The dataset is not included in the repository; share it only if its license and privacy status allow redistribution.

## Important interpretation note
This is an exploratory digital-wellness prototype, not a clinical assessment. Predicted probabilities may not be calibrated, dataset results may not generalize, and what-if simulations do not prove causal effects.

In [ ]:
from pathlib import Path
import pandas as pd

filename = "Smartphone_Usage_And_Addiction_Analysis_7500_Rows.csv"
file_path = Path(filename)
if not file_path.exists():
    raise FileNotFoundError(
        f"Required dataset not found: {file_path.resolve()}. "
        "Place the CSV beside this notebook or update filename."
    )

df = pd.read_csv(file_path)
required_columns = {
    "transaction_id", "user_id", "age", "gender",
    "daily_screen_time_hours", "social_media_hours", "gaming_hours",
    "work_study_hours", "sleep_hours", "notifications_per_day",
    "app_opens_per_day", "weekend_screen_time", "stress_level",
    "academic_work_impact", "addiction_level", "addicted_label"
}
missing = required_columns.difference(df.columns)
if missing:
    raise ValueError(f"Dataset is missing required columns: {sorted(missing)}")

print(f"Dataset shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
display(df.head())
display(df.isna().sum().to_frame("Missing values"))
display(df["addicted_label"].value_counts(dropna=False).to_frame("Count"))

## Train and evaluate

The preprocessing steps are fitted inside a scikit-learn pipeline after the stratified train/test split to avoid fitting transformers on the test set.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score

target = "addicted_label"
excluded_columns = ["transaction_id", "user_id", "addiction_level", target]
X = df.drop(columns=excluded_columns)
y = df[target]

numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])
model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=1000, class_weight="balanced", random_state=42
    ))
])

model.fit(X_train, y_train)
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, list(model.classes_).index(1)]

print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("ROC-AUC:", round(roc_auc_score(y_test, y_prob), 4))
print(classification_report(y_test, y_pred, zero_division=0))
print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))

## Permutation feature importance

This measures how much test-set ROC-AUC changes when one feature is shuffled. It is a model-interpretation aid, not evidence of causation.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.inspection import permutation_importance

importance_result = permutation_importance(
    model, X_test, y_test, scoring="roc_auc",
    n_repeats=5, random_state=42, n_jobs=-1
)
importance_df = pd.DataFrame({
    "Feature": X_test.columns,
    "Importance": importance_result.importances_mean,
    "Std": importance_result.importances_std
}).sort_values("Importance", ascending=False)

display(importance_df)
top = importance_df.head(10).sort_values("Importance")
plt.figure(figsize=(9, 5))
plt.barh(top["Feature"], top["Importance"], xerr=top["Std"])
plt.xlabel("Decrease in ROC-AUC when feature is shuffled")
plt.title("DigitalWell AI — Permutation Feature Importance")
plt.tight_layout()
plt.show()

## What-if habit simulation

Each scenario edits a test user's profile and sends it through the same fitted model. The resulting change is a model response to hypothetical inputs, not a causal estimate of a real intervention.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

baseline = X_test.iloc[[0]].copy()
def predict_probability(profile):
    return float(model.predict_proba(profile)[0][list(model.classes_).index(1)]) * 100

scenarios = {
    "Current habits": baseline.copy(),
    "Screen time -2h": baseline.copy(),
    "More sleep (+1h)": baseline.copy(),
    "Less social media (-1h)": baseline.copy()
}
scenarios["Screen time -2h"]["daily_screen_time_hours"] = max(
    0, float(baseline["daily_screen_time_hours"].iloc[0]) - 2
)
scenarios["More sleep (+1h)"]["sleep_hours"] = min(
    24, float(baseline["sleep_hours"].iloc[0]) + 1
)
scenarios["Less social media (-1h)"]["social_media_hours"] = max(
    0, float(baseline["social_media_hours"].iloc[0]) - 1
)

results_df = pd.DataFrame([
    {"Scenario": name, "Model-estimated score (%)": round(predict_probability(profile), 2)}
    for name, profile in scenarios.items()
])
display(results_df)
ax = results_df.plot(
    x="Scenario", y="Model-estimated score (%)",
    kind="bar", legend=False, figsize=(9, 5), ylim=(0, 100)
)
ax.set_ylabel("Model-estimated probability (%)")
ax.set_title("DigitalWell AI — Digital Habit Scenarios")
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.show()

## Export model artifacts

This saves the fitted preprocessing-plus-classification pipeline and feature metadata. The dashboard expects digitalwell_model.pkl beside app.py. Review the dataset and model licensing before sharing generated artifacts.

In [ ]:
import joblib
import json

joblib.dump(model, "digitalwell_model.pkl")
feature_info = {}
for column in X.columns:
    if pd.api.types.is_numeric_dtype(X[column]):
        feature_info[column] = {
            "type": "numeric",
            "min": float(X[column].min()),
            "max": float(X[column].max()),
            "default": float(X[column].median())
        }
    else:
        feature_info[column] = {
            "type": "categorical",
            "options": sorted(X[column].dropna().astype(str).unique().tolist()),
            "default": str(X[column].mode()[0])
        }

with open("digitalwell_features.json", "w", encoding="utf-8") as f:
    json.dump(feature_info, f, indent=2)

print("Saved digitalwell_model.pkl and digitalwell_features.json")
print("Model input columns:", list(X.columns))